# PIA05: Problema 2 - Detección de Materiales en Puentes

**Objetivo**: Segmentación Semántica.
Dado un puente, predecir el material (hormigón, metal, decorativo) píxel a píxel.
Usaremos **FastAI v2** y la librería auxiliar **SemTorch** (semejante al cuadernillo 504).

## 1. Setup: Librerías y Rutas

In [ ]:
# En Colab: Clonar repositorio con dataset + docs
import os
from pathlib import Path

repo_url = "https://github.com/kachytronico/PIA_tarea_05_dataset.git"
if os.path.exists('/content'):
    repo_dir = Path('/content/PIA_tarea_05_dataset')
else:
    repo_dir = Path('f:/Datos_alf_ SDD1/OneDrive - Diabeticos Asociados Riojanos/Documentos/GitHub/PIA_tarea_05_dataset')

if not repo_dir.exists() and os.path.exists('/content'):
    print("Clonando repo en Colab...")
    os.system(f"git clone {repo_url} {repo_dir}")

print(f"✅ Repositorio listo en: {repo_dir}")
print(f"📂 Contenido:\n")
for item in sorted(repo_dir.glob('*')):
    print(f"  {item.name}/") if item.is_dir() else print(f"  {item.name}")

In [ ]:
!pip install fastai semtorch numpy -Uqqq

from fastai.vision.all import *
import torch
from pathlib import Path

# Habilitar SemTorch
import semtorch
from semtorch import get_segmentation_learner

set_seed(42)
print(f"GPU disponible: {torch.cuda.is_available()}")

# Detectar ruta local o Colab
if os.path.exists('/content'):
    repo_root = Path('/content/PIA_tarea_05_dataset')
else:
    repo_root = Path('f:/Datos_alf_ SDD1/OneDrive - Diabeticos Asociados Riojanos/Documentos/GitHub/PIA_tarea_05_dataset')
path = repo_root / 'Material Detection' / 'Train'

print(f"Ruta del dataset: {path}")
path_images = path / 'images'
path_masks = path / 'masks'


## 2. Preparar los Datos (DataLoaders)
La mascarilla debe asociarse a cada imagen. Las etiquetas son típicamente colores o canales indexados.

In [ ]:
# Función para encontrar la máscara correspondiente a una imagen
def get_msk(fn): 
    # Asume que si la imagen es puente_01.jpg, la máscara es puente_01.png en la otra carpeta
    return path_masks / f'{fn.stem}.png'

# Los materiales de los puentes
codes = np.loadtxt(path / 'codes.txt', dtype=str) if (path/'codes.txt').exists() else ['Fondo', 'Hormigon', 'Metal', 'MetalDecorativo']
print(f"Códigos de clase detectados de ejemplo: {codes}")

db = DataBlock(
    blocks=(ImageBlock, MaskBlock(codes)),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=get_msk,
    item_tfms=Resize(256),
    batch_tfms=aug_transforms() # Incluye Data Augmentation geométrica (flips) y fotométrica (brillo/contraste)
)

# Usaremos un bs (batch size) adecuado para Google Colab T4 a est resolución.
dls = db.dataloaders(path_images, bs=8)
dls.show_batch(max_n=4, figsize=(10,7))


## 3. Modelo Básico de Segmentación (SemTorch Baseline)

In [ ]:
from semtorch import get_segmentation_learner

# Creamos un modelo base U-Net con encoder ResNet18 
# NOTA: Se empleará Dice y Exactitud como métricas.
learn = get_segmentation_learner(dls=dls, number_classes=len(codes), architecture=semtorch.models.unet, backbone_name='resnet18', metrics=[Dice(), accuracy_macrom])

learn.lr_find()


## 4. Entrenamiento con LR Óptimo y Prevención de Sobreajuste
Añado **EarlyStoppingCallback** y **Weight Decay (wd)** para un entrenamiento limpio que corte si la red empieza a memorizar en lugar de aprender.


In [ ]:
lr_optimo = 1e-3 # Este valor dependerá de la gráfica generada por lr_find()
callbacks = [EarlyStoppingCallback(patience=4)]

learn.fine_tune(10, base_lr=lr_optimo, wd=0.1, cbs=callbacks)
learn.save('baseline_seg')
learn.show_results(max_n=4, figsize=(12, 10))


## 5. Progressive Learning y Data Augmentation Total
El Augment ya lo añadí en `batch_tfms` al crear el bloque. Ahora escalo la imagen para detalles más finos.

In [ ]:
print("Escalando la resolución (Progressive Learning)...")
db_highres = DataBlock(
    blocks=(ImageBlock, MaskBlock(codes)),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=get_msk,
    item_tfms=Resize(512), # AUMENTO DE RESOLUCIÓN DE 256 a 512
    batch_tfms=aug_transforms()
)

dls_highres = db_highres.dataloaders(path_images, bs=4) # Reducimos batch size para evitar OutOfMemory
learn.dls = dls_highres

learn.fine_tune(4, base_lr=lr_optimo/10, wd=0.1, cbs=callbacks)
learn.save('progressive_seg')


## 6. Probar otra Arquitectura: ResNet34
Vamos a instanciar otra red U-Net pero esta vez le pondremos un "cerebro" (encoder) más profundo (`resnet34`).

In [ ]:
learn_34 = get_segmentation_learner(dls=dls_highres, number_classes=len(codes), architecture=semtorch.models.unet, backbone_name='resnet34', metrics=[Dice(), accuracy_macrom])
learn_34.fine_tune(4, base_lr=lr_optimo, wd=0.1, cbs=callbacks)


## 7. Matriz de Confusión y Conclusión

In [ ]:
interp = ClassificationInterpretation.from_learner(learn_34)
interp.plot_confusion_matrix(figsize=(8,8))

print("✅ Conclusión de la matriz:")
print("Aquí explicaremos si la red confunde hormigón con metal y su sensibilidad para el cliente.")
